# 09 · Modelo final: única evaluación en TEST
Completar `FINAL` con las decisiones de los notebooks 03, 05, 06 y 07 (todas tomadas en Validación). Este notebook (1) reentrena con 3 semillas, (2) calibra el umbral de alerta en Validación, (3) evalúa **una sola vez** en Test, (4) compara con persistencia y climatología con bootstrap por días y Diebold–Mariano, y (5) exporta los artefactos de la app.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
import json, joblib
from s4.pipeline import preparar_datos, correr_variante, baselines, calibrar_umbral, evaluar_y_registrar
from s4.train import configurar_gpu, predecir
from s4 import features as ft, evaluate as ev, stats
configurar_gpu()
FINAL = {"ventanas.lookback": 360, "perdida.alpha": 1.5, "perdida.beta": 100.0, "modelo.tipo": "apilado"}   # <-- decisiones
CONJUNTO = "E_completo"                                                                               # <-- decisión
cfg = variante(cargar_config('../config/ablacion_2025_2026.yaml'), **FINAL)
datos = preparar_datos(cfg)

In [ ]:
filas, modelos, parts = correr_variante(cfg, datos, ft.CONJUNTOS[CONJUNTO], "FINAL", "final_val", guardar_modelo=True)
mejor = int(np.argmax([f["HSS"] for f in filas]))          # semilla elegida en VALIDACIÓN
semilla, modelo, yhat_val = modelos[mejor]
esc = parts["escalador"]
u, tabla_u = calibrar_umbral(cfg, parts["val"], yhat_val)
print("Semilla:", semilla, "| umbral de alerta:", u)

In [ ]:
# ===== ÚNICA EVALUACIÓN EN TEST =====
pt = parts["test"]
yhat = predecir(modelo, pt, esc)
ref = baselines(cfg, datos, parts, "test")
filas_t = [evaluar_y_registrar(cfg, pt, ref["Persistencia"], "Persistencia", "final_test"),
           evaluar_y_registrar(cfg, pt, ref["Climatología diaria"], "Climatología", "final_test"),
           evaluar_y_registrar(cfg, pt, yhat, "Modelo final", "final_test", semilla, umbral_alerta=u)]
final = pd.DataFrame(filas_t)
final.to_csv(ruta(cfg,"reportes")/"tables/tabla_final_test.csv", index=False)
final[["modelo","RMSE_global","RMSE_evento","POD","Precision","FAR","HSS","POD_inicio","antelacion_media_min"]].round(4)

In [ ]:
m = ~pt.objetivo_reconstruido
for nombre, yref in ref.items():
    a = (pt.y_real, yref, yhat, pt.dia)
    print(f"--- Modelo final vs {nombre}")
    print("  RMSE evento:", stats.bootstrap_bloques(*a, stats.m_rmse_evento(), mascara=m))
    print("  RMSE global:", stats.bootstrap_bloques(*a, stats.m_rmse_global, mascara=m))
    print("  Diebold–Mariano:", stats.diebold_mariano(*a, mascara=m))
display(ev.metricas_por_fase(pt, yhat, umbral_alerta=u))
display(ev.analisis_falsas_alarmas(pt, yhat, umbral_alerta=u))

In [ ]:
# Exportar artefactos para la app (mismo formato que espera app.py + campos nuevos)
DEPLOY = ruta(cfg, "base") / "app"; DEPLOY.mkdir(exist_ok=True)
modelo.save(DEPLOY / "modelo_S4_lstm.keras", include_optimizer=False)
joblib.dump(esc, DEPLOY / "SCALER_S4_app.pkl")
fila = final[final["modelo"] == "Modelo final"].iloc[0]
json.dump({"LOOKBACK": cfg["ventanas"]["lookback"], "HORIZON": cfg["ventanas"]["horizon"],
           "THRESHOLD": cfg["datos"]["umbral_evento"], "UMBRAL_ALERTA": u,
           "FEATURES_COLS": esc.columnas_, "TARGET_COL": "S4", "SAMPLING_MINUTES": 1,
           "ARQUITECTURA": cfg["modelo"]["tipo"], "UNITS1": cfg["modelo"].get("units1"), "UNITS2": cfg["modelo"].get("units2"),
           "RMSE_EVENTO_TEST": float(fila["RMSE_evento"]), "HSS_TEST": float(fila["HSS"]),
           "POD_TEST": float(fila["POD"]), "PRECISION_TEST": float(fila["Precision"]), "SEMILLA": int(semilla)},
          open(DEPLOY / "config.json", "w"), indent=2, ensure_ascii=False)
print("Artefactos exportados en", DEPLOY)